In [2]:
from Class_PAS_Data_Extract import PASDataEngine
from Class_PAS_Product import Product
import Class_PAS_Graph
import importlib
import pandas as pd

In [ ]:
plotPaths = "plots/"
key_file = r'\\shuser-Prod.intel.com\shProduser$\dagarcia\keys\secret.key'
credential_file = r'\\shuser-Prod.intel.com\shProduser$\dagarcia\EncryptedCredentials\credentials.encrypted'


Server = "smtpauth.intel.com"
Port = "587"
UserEmail = "david.a.garcia@intel.com"


In [3]:
def read_excel_to_dataframe(file_path, sheet_name, halt_on_error=True):
    """
    Reads a specific worksheet from an Excel file into a Pandas DataFrame.

    Parameters:
    - file_path (str): The path to the Excel file.
    - sheet_name (str or int): The name or index of the worksheet to load.
    - halt_on_error (bool): Flag indicating whether to halt execution on error.
                            If False, the function will return None on error.

    Returns:
    - DataFrame or None: The DataFrame if successful, or None if an error occurs and halt_on_error is False.
    """
    try:
        # Attempt to read the specified worksheet into a DataFrame
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        return df
    except Exception as e:
        # Handle the error based on the halt_on_error flag
        if halt_on_error:
            print(f"Error reading {file_path}, sheet {sheet_name}: {e}")
            raise  # Re-raise the exception to halt execution
        else:
            print(f"Error reading {file_path}, sheet {sheet_name}: {e}. Returning None.")
            return None



In [4]:
config_file = r'\\azshfs.intel.com\AZAnalysis$\1272_MAODATA\Config\PDE\dagarcia\PAS_CONFIG\P1275_Config.xlsx'

pas_config = read_excel_to_dataframe(config_file, 'PlotConfig', halt_on_error=True)
email_config = read_excel_to_dataframe(config_file, 'EmailConfig', halt_on_error=True)
reticle_config = read_excel_to_dataframe(config_file, 'ReticleConfig', halt_on_error=False)

In [5]:
if  pas_config is not None:
    unique_combos = pas_config.groupby(['PRODUCT', 'FAB_PROD', 'RET_PROD'])['COMMIT'].max().reset_index()

    products = unique_combos.set_index('PRODUCT').to_dict(orient='index')
    
    for product, details in products.items():
        print(f"Processing Product: {product}")
        print(f"FAB_PROD: {details['FAB_PROD']}, RET_PROD: {details['RET_PROD']}, COMMIT: {details['COMMIT']}")



Processing Product: Sundance Mesa 1
FAB_PROD: 8PWQCVA, RET_PROD: 8PWQCA, COMMIT: 2025-07-28 18:00:00


In [6]:
# products['Sundance Mesa 1']
pas_config

,PRODUCT,TITLE,LOT,COMMIT,FAB_PROD,RET_PROD,ORIG_COMMIT
0,Sundance Mesa 1,Lead Lot,L5138740,2025-07-28 18:00:00,8PWQCVA,8PWQCA,2025-08-14
1,Sundance Mesa 1,Scout 1,L5139550,2025-07-27 18:00:00,8PWQCVA,8PWQCA,2025-08-04
2,Sundance Mesa 1,Scout 2,L5139560,2025-07-27 18:00:00,8PWQCVA,8PWQCA,2025-08-04


In [7]:
dataengine = PASDataEngine()

In [8]:

prod = Product('8PWQCA', products['Sundance Mesa 1'],  dataengine, ret_version=reticle_config, lots=None, debug_flag=True)

d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Product.py:118: FutureWarning: The provided callable <function min at 0x000001414CEEE480> is currently using DataFrameGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  RetData = pd.pivot_table(RetData,index=['LAYER'], values=['TAPEIN_TREND','ITO_TREND','FAB_REQUIREDDATE','IMO_TREND','SHIPDATE'], aggfunc=np.min).reset_index()


In [9]:
prod.add_lot('L5138740', 'Lead Lot')
prod.add_lot('L5139550', 'Scout 1')
prod.add_lot('L5139560', 'Scout 2')

d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string

In [10]:
prod.build_plot_data()


Difference in days: 158 days
Rounded difference: 161.0 days
ymin_date: 2025-02-22 00:00:00
ymin_val: 0
ymax_date: 2025-08-02 00:00:00
ymax_val: 161.0


In [18]:


importlib.reload(importlib.import_module('Class_PAS_Graph'))
myGraph = Class_PAS_Graph.PASPlot(prod,plotPaths)
myGraph.make_plot()

commit_date: 2025-07-28 18:00:00
trend_date: 2025-07-23 17:13:27.203438368


d:\Python\PAS_Graphs\Class_PAS_Graph.py:106: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax2.set_yticklabels([])


In [ ]:


debug=True

['\\\\shuser-Prod.intel.com\\shProduser$\\dagarcia\\PAS\\base.ini']

In [33]:
email_addresses = email_config['Email'].to_list()
if debug:
    email_addresses = email_config.loc[email_config['Debug'] == True, 'Email'].to_list()
    
email_addresses



['david.a.garcia@intel.com']

In [36]:
body = ""

msg = EmailMessage()
msg['Subject'] = 'PAS Data'
msg['From'] = UserEmail
msg['To'] = ', '.join(email_addresses)
msg.set_content('Please find the attached PAS data plots.')

html_body = f"<html><body><p>{body}</p>"

directory = Path(plotPaths)